# 01 - Veri seti envanteri
Fit3D ve Human3.6M erişimini, lisans notlarını, kişi/deneme/kamera kimliklerini ve referans eşleşmesini denetler. `00_environment_check.ipynb` bootstrap hücresini önce çalıştırın.

In [ ]:
import sys, zipfile
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    artifacts_root = Path('/content/drive/MyDrive/PhysioLoad_AI_Project')
    code_root = Path('/content/physioload-ai')
    if not (code_root / 'physioload_env.py').exists():
        code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(artifacts_root / 'code' / 'physioload-source.zip') as package: package.extractall(code_root)
else:
    artifacts_root = None
    here = Path.cwd().resolve()
    code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))
from physioload_env import setup_environment
env = setup_environment(root=code_root, artifacts_root=artifacts_root)
print(env.summary())

In [ ]:
from pathlib import Path
import pandas as pd

columns = ['image_id','dataset','subject_id','trial_id','camera_id','image_path','pose_label','reference_path','view_label','frame_index']
inventory_path = env.metadata_dir / 'dataset_inventory.csv'
if inventory_path.exists():
    inventory = pd.read_csv(inventory_path)
else:
    inventory = pd.DataFrame(columns=columns)
    inventory.to_csv(inventory_path, index=False)
print('inventory:', inventory_path, 'rows:', len(inventory))
inventory.head()

In [ ]:
required = {'image_id','dataset','subject_id','trial_id','camera_id','image_path'}
missing = required - set(inventory.columns)
if missing:
    raise ValueError(f'Eksik sütunlar: {sorted(missing)}')
summary = inventory.groupby('dataset', dropna=False).agg(
    images=('image_id','nunique'), subjects=('subject_id','nunique'), trials=('trial_id','nunique'), cameras=('camera_id','nunique')
).reset_index() if len(inventory) else pd.DataFrame()
summary

## Çıkış kapısı
Aşağıdaki üç koşul sağlanmadan sonraki notebook'lara geçmeyin: resmi erişim/lisans kaydı, görüntü-referans eşleşmesi ve en az üç uygun statik duruş adayı.